# 类别型特征分箱示例

本示例展示如何对类别型特征进行自定义分箱。

## 类别型特征分箱的特点

1. **自动识别**: 根据 dtype 自动识别类别型特征 (object/string/category)
2. **自定义规则**: 支持通过 `user_splits` 参数自定义分箱规则
3. **类别合并**: 支持将多个类别合并到同一个分箱
4. **特殊值处理**: 支持单独处理特殊值和缺失值
5. **List[List]格式**: 支持类似toad的List[List]格式存储类别分箱规则

## 1. 环境准备

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# 添加项目路径
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from hscredit.core.binning import OptimalBinning
from hscredit.report import feature_bin_stats

print("✅ 模块导入成功")

✅ 模块导入成功


## 2. 创建示例数据

In [2]:
# 创建包含类别型特征的示例数据
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    '城市': np.random.choice(
        ['北京', '上海', '广州', '深圳', '杭州', '成都', '武汉', '西安'],
        n_samples
    ),
    '学历': np.random.choice(
        ['高中', '大专', '本科', '硕士', '博士'],
        n_samples
    ),
    '职业': np.random.choice(
        ['公务员', '教师', '医生', '工程师', '销售', '自由职业', '其他'],
        n_samples
    ),
    '收入等级': np.random.choice(
        ['低', '中', '高'],
        n_samples
    ),
    '年龄': np.random.randint(18, 65, n_samples),
    'target': np.random.randint(0, 2, n_samples)
})

# 添加一些缺失值
df.loc[df.sample(50).index, '城市'] = np.nan
df.loc[df.sample(30).index, '学历'] = np.nan

print(f"数据集形状: {df.shape}")
print(f"\n前5行:")
print(df.head())

print(f"\n各列的数据类型:")
print(df.dtypes)

数据集形状: (1000, 6)

前5行:
   城市   学历  职业 收入等级  年龄  target
0  武汉   高中  教师    高  24       0
1  深圳   高中  其他    高  20       1
2  杭州   高中  教师    高  18       1
3  武汉   博士  教师    中  53       1
4  广州  NaN  其他    中  42       0

各列的数据类型:
城市        object
学历        object
职业        object
收入等级      object
年龄         int64
target     int64
dtype: object


## 3. 自动识别类别型特征

In [3]:
# OptimalBinning 会自动识别类别型特征
# 使用tree方法（支持类别型变量）
binner = OptimalBinning(method='tree', max_n_bins=5)

X = df[['城市', '学历', '职业', '收入等级', '年龄']]
y = df['target']

binner.fit(X, y)

# 查看特征类型识别结果
print("特征类型识别结果:")
print("="*60)
for feature, ftype in binner.feature_types_.items():
    print(f"{feature:<15} {ftype:>12}")

特征类型识别结果:
城市               categorical
学历               categorical
职业               categorical
收入等级             categorical
年龄                 numerical


## 4. 类别型特征自定义分箱 - List[List]格式（推荐）

In [4]:
# 方法1: 使用List[List]格式（推荐，类似toad）
# 每个元素是一个列表，包含该分箱的所有类别值
# 最后一个列表可以包含np.nan表示缺失值

user_splits = {
    '城市': [
        ['北京', '上海'],           # 第1箱: 北京和上海
        ['广州', '深圳', '杭州'],    # 第2箱: 广州、深圳、杭州
        ['成都', '武汉', '西安'],    # 第3箱: 成都、武汉、西安
        [np.nan]                    # 第4箱: 缺失值（可选）
    ],
    '学历': [
        ['高中', '大专'],           # 第1箱: 高中和大专
        ['本科'],                   # 第2箱: 本科
        ['硕士', '博士'],           # 第3箱: 硕士和博士
        [np.nan]                    # 第4箱: 缺失值（可选）
    ]
}

binner = OptimalBinning(user_splits=user_splits)
binner.fit(X, y)

# 查看分箱结果
for feature in ['城市', '学历']:
    print(f"\n【{feature}】List[List]格式分箱结果:")
    print("="*80)
    bin_table = binner.get_bin_table(feature)
    print(bin_table[['分箱标签', '样本总数', '坏样本率', '分档WOE值']].to_string(index=False))


【城市】List[List]格式分箱结果:
    分箱标签  样本总数     坏样本率    分档WOE值
   北京,上海   237 0.514768  0.051089
广州,深圳,杭州   361 0.523546  0.086252
成都,武汉,西安   352 0.474432 -0.110362
     nan    50 0.480000 -0.088043

【学历】List[List]格式分箱结果:
 分箱标签  样本总数     坏样本率    分档WOE值
高中,大专   406 0.482759 -0.076993
   本科   204 0.500000 -0.008000
硕士,博士   360 0.516667  0.058691
  nan    30 0.600000  0.397465


## 5. 导出和导入List[List]格式规则

In [5]:
# 导出规则
rules = binner.export_rules()

print("导出的分箱规则:")
print("="*80)
for feature, rule in rules.items():
    print(f"{feature}:")
    if isinstance(rule, list) and len(rule) > 0 and isinstance(rule[0], list):
        # List[List]格式
        for group in rule:
            print(f"  - {group}")
    else:
        # 数值型格式
        print(f"  - {rule}")
    print()

# 导入规则到新的分箱器
binner_new = OptimalBinning()
binner_new.import_rules(rules)

# 应用分箱
print("="*80)
print("导入规则后重新应用分箱:")
print("="*80)

binner_new.fit(X[['城市', '学历']], y)

bin_table_original = binner.get_bin_table('城市')
bin_table_new = binner_new.get_bin_table('城市')

print(f"\n【城市】导入规则后的分箱结果:")
print("="*80)
print(bin_table_new[['分箱标签', '样本总数', '坏样本率', '分档WOE值']].to_string(index=False))

# 验证一致性
if bin_table_original['样本总数'].tolist() == bin_table_new['样本总数'].tolist():
    print("\n验证: 导出-导入循环一致性 ✅")

导出的分箱规则:
城市:
  - ['北京', '上海']
  - ['广州', '深圳', '杭州']
  - ['成都', '武汉', '西安']
  - [nan]

学历:
  - ['高中', '大专']
  - ['本科']
  - ['硕士', '博士']
  - [nan]

职业:
  - []

收入等级:
  - []

年龄:
  - [27.0, 36.0, 45.0, 55.0]

导入规则后重新应用分箱:

【城市】导入规则后的分箱结果:
    分箱标签  样本总数     坏样本率    分档WOE值
   北京,上海   237 0.514768  0.051089
广州,深圳,杭州   361 0.523546  0.086252
成都,武汉,西安   352 0.474432 -0.110362
     nan    50 0.480000 -0.088043

验证: 导出-导入循环一致性 ✅


## 6. 向后兼容 - 字符串逗号分隔格式

In [6]:
# 方法2: 使用字符串逗号分隔格式（向后兼容）
# 每个元素是一个字符串，多个类别用逗号分隔

user_splits_old = {
    '城市': [
        '北京,上海',           # 第1箱: 北京和上海
        '广州,深圳,杭州',      # 第2箱: 广州、深圳、杭州
        '成都,武汉,西安'       # 第3箱: 成都、武汉、西安
    ],
    '学历': [
        '高中,大专',           # 第1箱: 高中和大专
        '本科',                # 第2箱: 本科
        '硕士,博士'            # 第3箱: 硕士和博士
    ]
}

binner_old = OptimalBinning(user_splits=user_splits_old, missing_separate=True)
binner_old.fit(X, y)

# 查看分箱结果
for feature in ['城市']:
    print(f"\n【{feature}】字符串格式分箱结果:")
    print("="*80)
    bin_table = binner_old.get_bin_table(feature)
    print(bin_table[['分箱标签', '样本总数', '坏样本率', '分档WOE值']].to_string(index=False))


【城市】字符串格式分箱结果:
                分箱标签  样本总数     坏样本率    分档WOE值
       (-inf, 北京,上海]   237 0.514768  0.051089
   (北京,上海, 广州,深圳,杭州]   361 0.523546  0.086252
(广州,深圳,杭州, 成都,武汉,西安]   352 0.474432 -0.110362
             missing    50 0.480000 -0.088043


## 7. 应用分箱转换

In [7]:
# 创建包含数值型和类别型的分箱规则
mixed_rules = {
    '城市': [
        ['北京', '上海'],
        ['广州', '深圳', '杭州'],
        ['成都', '武汉', '西安'],
        [np.nan]
    ],
    '学历': [
        ['高中', '大专'],
        ['本科'],
        ['硕士', '博士'],
        [np.nan]
    ],
    '年龄': [25, 35, 45, 55]  # 数值型: 切分点
}

binner_mixed = OptimalBinning(user_splits=mixed_rules)
binner_mixed.fit(X[['城市', '学历', '年龄']], y)

# 应用分箱转换
test_data = X[['城市', '学历', '年龄']].head()

print("原始数据:")
print(test_data)

# 转换为分箱索引
binned_indices = binner_mixed.transform(test_data, metric='indices')
print("\n分箱后（indices）:")
print(binned_indices)

# 转换为分箱标签
binned_labels = binner_mixed.transform(test_data, metric='bins')
print("\n分箱后（bins）:")
print(binned_labels)

原始数据:
   城市   学历  年龄
0  武汉   高中  24
1  深圳   高中  20
2  杭州   高中  18
3  武汉   博士  53
4  广州  NaN  42

分箱后（indices）:
   城市  学历  年龄
0   2   0   0
1   1   0   0
2   1   0   0
3   2   2   3
4   1   3   2

分箱后（bins）:
         城市     学历              年龄
0  成都,武汉,西安  高中,大专   (-inf, 25.00]
1  广州,深圳,杭州  高中,大专   (-inf, 25.00]
2  广州,深圳,杭州  高中,大专   (-inf, 25.00]
3  成都,武汉,西安  硕士,博士  (45.00, 55.00]
4  广州,深圳,杭州    nan  (35.00, 45.00]


## 8. JSON序列化（保存和加载规则）

In [8]:
import json

# 导出规则
rules = binner_mixed.export_rules()

# 将np.nan转换为字符串"NaN"以便JSON序列化
def convert_nan_to_str(obj):
    if isinstance(obj, dict):
        return {k: convert_nan_to_str(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_nan_to_str(item) for item in obj]
    elif isinstance(obj, float) and np.isnan(obj):
        return "NaN"
    return obj

rules_json = convert_nan_to_str(rules)

print("JSON格式的分箱规则:")
print("="*80)
print(json.dumps(rules_json, indent=2, ensure_ascii=False))

# 将字符串"NaN"转换回np.nan
def convert_str_to_nan(obj):
    if isinstance(obj, dict):
        return {k: convert_str_to_nan(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_str_to_nan(item) for item in obj]
    elif obj == "NaN":
        return np.nan
    return obj

# 模拟从JSON文件加载
rules_loaded = convert_str_to_nan(rules_json)

# 导入到新的分箱器
binner_loaded = OptimalBinning()
binner_loaded.import_rules(rules_loaded)

print("\n✅ JSON序列化和反序列化成功")

JSON格式的分箱规则:
{
  "城市": [
    [
      "北京",
      "上海"
    ],
    [
      "广州",
      "深圳",
      "杭州"
    ],
    [
      "成都",
      "武汉",
      "西安"
    ],
    [
      "NaN"
    ]
  ],
  "学历": [
    [
      "高中",
      "大专"
    ],
    [
      "本科"
    ],
    [
      "硕士",
      "博士"
    ],
    [
      "NaN"
    ]
  ],
  "年龄": [
    25,
    35,
    45,
    55
  ]
}

✅ JSON序列化和反序列化成功


## 9. 完整示例: 真实业务场景

In [9]:
# 场景: 信贷风控中的特征分箱
# 根据业务知识和风险表现定义分箱规则

# 1. 定义业务分箱规则（使用List[List]格式）
business_rules = {
    '城市': [
        ['北京', '上海', '广州', '深圳'],  # 一线城市(优质客户)
        ['杭州', '成都', '武汉', '西安'],   # 新一线/二线城市
        [np.nan]                            # 缺失值
    ],
    '学历': [
        ['博士', '硕士'],  # 高学历
        ['本科'],          # 中等学历
        ['大专', '高中'],  # 低学历
        [np.nan]           # 缺失值
    ],
    '职业': [
        ['公务员', '医生', '教师'],           # 稳定职业
        ['工程师'],                           # 技术职业
        ['销售', '自由职业', '其他'],          # 其他职业
    ],
    '年龄': [25, 35, 45, 55],  # 数值型: 切分点
}

# 2. 应用分箱
binner = OptimalBinning(
    user_splits=business_rules,
    missing_separate=True
)

binner.fit(X, y)

# 3. 查看所有特征的分箱结果
print("业务分箱结果汇总:")
print("="*80)
print(f"{'特征':<15} {'IV值':>10} {'分箱数':>8}")
print("-"*80)

for feature in ['城市', '学历', '职业', '年龄']:
    bin_table = binner.get_bin_table(feature)
    iv = bin_table['分档IV值'].sum()
    n_bins = len(bin_table)
    print(f"{feature:<15} {iv:>10.4f} {n_bins:>8}")

# 4. 导出分箱规则供后续使用
rules = binner.export_rules()
print(f"\n导出的分箱规则:")
print("="*80)
rules_json = convert_nan_to_str(rules)
print(json.dumps(rules_json, indent=2, ensure_ascii=False))

业务分箱结果汇总:
特征                     IV值      分箱数
--------------------------------------------------------------------------------
城市                  0.0021        3
学历                  0.0083        4
职业                  0.0023        3
年龄                  0.0043        5

导出的分箱规则:
{
  "城市": [
    [
      "北京",
      "上海",
      "广州",
      "深圳"
    ],
    [
      "杭州",
      "成都",
      "武汉",
      "西安"
    ],
    [
      "NaN"
    ]
  ],
  "学历": [
    [
      "博士",
      "硕士"
    ],
    [
      "本科"
    ],
    [
      "大专",
      "高中"
    ],
    [
      "NaN"
    ]
  ],
  "职业": [
    [
      "公务员",
      "医生",
      "教师"
    ],
    [
      "工程师"
    ],
    [
      "销售",
      "自由职业",
      "其他"
    ]
  ],
  "收入等级": [],
  "年龄": [
    25,
    35,
    45,
    55
  ]
}


## 总结

### 类别型特征分箱的格式

#### 1. List[List]格式（推荐）
```python
user_splits = {
    '特征名': [
        ['类别1', '类别2'],  # 第一箱
        ['类别3'],           # 第二箱
        [np.nan]             # 缺失值箱（可选）
    ]
}
```

**优点**：
- 类似toad的实现方式
- 清晰明确，易于理解
- 支持np.nan缺失值
- 易于JSON序列化（需转换np.nan）

#### 2. 字符串逗号分隔格式（向后兼容）
```python
user_splits = {
    '特征名': [
        '类别1,类别2',  # 第一箱
        '类别3'         # 第二箱
    ]
}
```

### 关键功能

1. **自动识别**: 系统会自动识别 object/string/category 类型的特征

2. **导出/导入规则**:
   ```python
   # 导出
   rules = binner.export_rules()
   
   # 导入
   binner_new = OptimalBinning()
   binner_new.import_rules(rules)
   ```

3. **缺失值处理**: 
   - 在List[List]中包含 `[np.nan]` 单独分箱
   - 或使用 `missing_separate=True` 参数

4. **混合类型支持**: 可以同时处理数值型和类别型特征

5. **JSON序列化**: 需要将np.nan转换为字符串"NaN"后才能JSON序列化

### 最佳实践

- **业务理解优先**: 根据业务知识和风险表现定义分箱
- **WOE单调性**: 确保分箱后的WOE值有明显区分度
- **样本均衡**: 每个分箱的样本量不宜过少
- **缺失值单独处理**: 缺失值通常包含信息，建议单独分箱
- **使用List[List]格式**: 清晰且符合行业标准

### 与toad对比

| 特性 | hscredit | toad |
|------|----------|------|
| List[List]格式 | ✅ | ✅ |
| 缺失值单独分箱 | ✅ | ✅ |
| 数值型特征 | ✅ | ✅ |
| 导出/导入规则 | ✅ | ✅ |
| JSON序列化 | ✅ (需转换np.nan) | ✅ |